# LFM Instance Segmentation Example Workflow
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data. 

## Purpose of this notebook
This notebook is designed to be used as an example semantic segmentation workflow ("crater" vs "non crater" model prediction). A pretrained DinoV3 model is loaded from disk, and we build our own crater detection model on top of this. The model loads data from the lfm project space, then runs several epochs of training on this data, and finally visualizes the model performance on the validation dataset. If you would like to control some of the model parameters, see the "User Configuration" section below.  

**Note**: currently, the training dataset used in this notebooks is comprised of 12-band input data (5 VIS bands, 2 UV bands, 5 Kaguya static bands). If you wish to train your own model with a different input dataset (i.e. different static band configuration), that will be added soon. If you would like to filter out certain WAC/Static bands, you can filter them using the BAND_FILTER variable in the config. **See the README in the [LFM repo](https://github.com/nasa-nccs-hpda/lfm) for more info.** 

## Imports, Dino Repo Clone

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "0"

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import sys
from glob import glob
from pathlib import Path
from types import SimpleNamespace

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import torch


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "lfm").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the LFM repo root from the current working directory.")


repo_root = find_repo_root()
repo_root = Path(str(repo_root).replace("/panfs/ccds02/nobackup", "/explore/nobackup"))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.utils import create_timestamped_output_dir
from lfm.toy_model.inst_seg import instance_toy_components

print("Successfully imported LFM Lightning/OOP modules")


## User Configuration

#### Paths
`INPUT_ROOT_DIR`: there are multiple different dataset configurations under the lfm/model_inputs/300_300_inputs folder. Choose any of the band subdirectories in the 300_300_inputs folder, and leave the other variables under "data paths" (IMAGE_DIR, LABEL_DIR, etc) as they are. 

`OUTPUT_DIR`: this is a relative path, so by default the outputs (visualizations, model checkpoints, dataset statistics) will go to the same folder as the notebook. 

#### Dataset parameters
`MAX_SAMPLES`: number of training samples to look for in INPUT_ROOT_DIR. 

`TRAIN_SPLIT`: weight of how many training samples versus validation samples; the default is 0.8, or 80% training.

#### Training hyperparameters
`BATCH_SIZE`: best to leave this at 16 to conserve VRAM, especially at higher number of input bands (>7). 

`NUM_EPOCHS`: default is 100, the model tends to start to get its best results around here. Feel free to experiment with this.

`BASE_LR`,`WEIGHT_DECAY`: feel free to change these to adjust how aggressively the model tries to tune to each training batch. Higher means more aggressive model learning, but can also mean the model has a harder time converging to the correct result. 

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the Dino backbone frozen; we found better results with False, but feel free to try with freezing set to true. 

`NUM_BANDS`: number of bands to include in the input. Currently supported are 3/5/7/12-band inputs.

`BAND_FILTER`: list of band indices in the range [0, 11], 0-indexed, to use in the dataset. For example, if I want to use only VIS bands, I would supply [0, 1, 2, 3, 4]. Band ordering is the following for the 12-band dataset: VIS (0-4), UV (5, 6), KAGUYA STATIC (7-11). 

- <mark>Note: the toy model expects a minimum of 3 bands (RGB). Ensure the band filter has at least 3 vis bands, ideally vis bands [3, 1, 0] which are closest in wavelength to RGB. </mark>

In [ ]:
# Data paths. The Lightning/OOP datamodule expects:
#   INPUT_ROOT_DIR/train/chips, INPUT_ROOT_DIR/train/labels
#   INPUT_ROOT_DIR/val/chips, INPUT_ROOT_DIR/val/labels
#   INPUT_ROOT_DIR/test/chips, INPUT_ROOT_DIR/test/labels
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg"

OUTPUT_DIR = Path("./outputs/inst_seg")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

MAX_SAMPLES = 500
BATCH_SIZE = 16
NUM_EPOCHS = 1
BASE_LR = 5e-5
WEIGHT_DECAY = 1e-3
FREEZE_ENCODER = False
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
TOY_ARCHITECTURE = "mask2former"  # mask2former, dino-mask-rcnn, dino-terratorch-mask-rcnn

instance_config = SimpleNamespace(
    data_root=INPUT_ROOT_DIR,
    toy_architecture=TOY_ARCHITECTURE,
    toy_batch_size=BATCH_SIZE,
    toy_num_workers=8,
    target_size=256,
    image_glob="*.tif",
    label_glob="*_label.npz",
    image_suffix="_input_wac_static_chip",
    label_suffix="_label",
    band_filter=BAND_FILTER,
    toy_normalize_inputs=False,
    normalization_source="finetune",
    normalization_modality="vis_uv",
    mask_shift=(0, 0),
    no_data_replace=None,
    no_label_replace=None,
    ignore_nodata_in_loss=False,
    nodata_ignore_index=-1,
    max_train_samples=MAX_SAMPLES,
    max_val_samples=MAX_SAMPLES,
    max_test_samples=MAX_SAMPLES,
    dino_checkpoint=None,
    toy_freeze_backbone=FREEZE_ENCODER,
    toy_learning_rate=BASE_LR,
    toy_weight_decay=WEIGHT_DECAY,
    max_epochs=NUM_EPOCHS,
    toy_gradient_clip_val=1.0,
    graha_anchor_sizes=(32, 64, 128, 256, 512),
    graha_anchor_aspect_ratios=(0.5, 1.0, 2.0),
    plot_n_samples=5,
    plot_every_n_epochs=1,
    prediction_split="val",
    prediction_n_samples=5,
    prediction_score_threshold=0.5,
    progress_log_every_n_batches=5,
    run_epoch_test_suite=False,
    epoch_test_split="test",
    epoch_test_n_samples=5,
    epoch_test_every_n_epochs=1,
    toy_lightning_checkpoint=None,
    skip_toy_fit=False,
    seed=42,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Create output directory
This will contain model checkpoints and visualizations. It will ask you to overwrite the directory if it already exists, so ensure you copy files that you want to keep to a different directory!

In [ ]:
OUTPUT_DIR = create_timestamped_output_dir(OUTPUT_DIR)


## Create dataloaders

In [ ]:
print("\n" + "=" * 60)
print("STEP 1: Creating Lightning datamodule.")
print("=" * 60)

toy_datamodule = instance_toy_components.create_datamodule(instance_config)
weight_assignments = toy_datamodule.weight_assignments

print(f"Train batches: {len(toy_datamodule.train_dataloader())}")
print(f"Val batches: {len(toy_datamodule.val_dataloader())}")


## Load Encoder and Create Model

In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Creating Toy instance Lightning task.")
print("=" * 60)

toy_task = instance_toy_components.create_task(
    instance_config,
    weight_assignments,
)
toy_image_processor = instance_toy_components.create_image_processor(instance_config)


In [ ]:
print(type(toy_task))
print(f"Image processor: {type(toy_image_processor).__name__ if toy_image_processor is not None else None}")


## Run Training

In [ ]:
print("\n" + "=" * 60)
print("Starting Lightning training.")
print("=" * 60)

toy_trainer = instance_toy_components.create_trainer(
    instance_config,
    OUTPUT_DIR,
    toy_image_processor,
)
toy_trainer.fit(toy_task, datamodule=toy_datamodule)


## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = OUTPUT_DIR / "plots" / "single_model" / "toy_model"
visualization_filenames = sorted(glob(str(visualization_dir / "*.png")))
print(f"Found {len(visualization_filenames)} visualization file(s) in {visualization_dir}")


In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.show()

In [ ]:
for name in ["toy_trainer", "toy_task", "toy_datamodule", "toy_image_processor"]:
    if name in globals():
        del globals()[name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
